# FlashNorm realized speedup in HuggingFace Transformers (Colab A100)

Reproducibility notebook for the realized end-to-end speedup of FlashNorm Proposition 1 in stock HuggingFace Transformers.

**What we measure.** `LlamaRMSNorm.forward()` always computes `self.weight * x_normalized`, even when `self.weight` is the all-ones tensor that FlashNorm folding leaves behind. PyTorch's torch C++ kernel `F.rms_norm(x, shape, weight=None, eps)` already branches on `weight is None` and skips the per-channel multiply. We monkey-patch every `LlamaRMSNorm` module's `forward` to route through the weightless C++ path and measure the speedup.

Both runs use the same checkpoint (`open-machine/Llama-3.2-1B-FlashNorm`), so the gain isolates kernel choice from any other variable.

**Wall-clock:** ~3 min (model load + 2 variants × 23 generations of 256 tokens each).

**Runtime requirement:** Colab A100. `Runtime > Change runtime type > A100 GPU`.

## Setup

Install pattern matches `flashNorm_decode_profile.ipynb` (verified to run cleanly on Colab A100). No vLLM, no version pinning conflicts.

In [ ]:
import os, sys, time, json, gc, subprocess
from pathlib import Path

import torch
assert torch.cuda.is_available(), "Need a GPU. Runtime > Change runtime type > A100 GPU."
GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

print("installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                       "transformers", "accelerate", "sentencepiece", "protobuf"])
import transformers
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}, CUDA: {torch.version.cuda}")

## Method

HF Transformers' `LlamaRMSNorm.forward()` is:

```python
def forward(self, x):
    input_dtype = x.dtype
    x = x.to(torch.float32)
    variance = x.pow(2).mean(-1, keepdim=True)
    x = x * torch.rsqrt(variance + self.variance_epsilon)
    return self.weight * x.to(input_dtype)   # always multiplies, even if weight is all-ones
```

PyTorch's C++ `F.rms_norm(x, shape, weight=None, eps)` skips the per-channel multiply when `weight is None`. We replace each `LlamaRMSNorm.forward` at runtime with one that routes through the C++ kernel with `weight=None`, run the same generation, and compare wall-clock.

**Variant A** (baseline): default HF `LlamaRMSNorm.forward`.
**Variant B** (FlashNorm-realized): runtime monkey-patch to `F.rms_norm(x, shape, None, eps)`, fp32 cast preserved for numerical parity.

In [ ]:
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.llama.modeling_llama import LlamaRMSNorm

MODEL_ID    = "open-machine/Llama-3.2-1B-FlashNorm"
PROMPT      = "Write a short story about a robot learning to paint."
N_WARMUP    = 3
N_TRIALS    = 20
MAX_TOKENS  = 256

print(f"loading {MODEL_ID}...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda",
    attn_implementation="eager", trust_remote_code=True
).eval()
print(f"loaded in {time.time()-t0:.1f}s")

inputs = tokenizer(PROMPT, return_tensors="pt").to("cuda")

def benchmark(label):
    print(f"\n--- {label} ---")
    for _ in range(N_WARMUP):
        with torch.no_grad():
            _ = model.generate(**inputs, max_new_tokens=MAX_TOKENS, do_sample=False, use_cache=True)
        torch.cuda.synchronize()
    times = []
    for trial in range(N_TRIALS):
        torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_TOKENS, do_sample=False, use_cache=True)
        torch.cuda.synchronize()
        elapsed = time.time() - t0
        times.append(elapsed)
        if trial < 2 or (trial+1) % 5 == 0:
            print(f"  trial {trial+1:3d}: {elapsed:.2f}s ({MAX_TOKENS/elapsed:.2f} tok/s)")
    median_s = sorted(times)[len(times)//2]
    median_tps = MAX_TOKENS / median_s
    sample = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)[:120]
    print(f"  median: {median_s:.2f}s -> {median_tps:.2f} tok/s")
    return median_s, median_tps, sample

# Variant A: HF default forward
ms_a, tps_a, sample_a = benchmark("A: HF default LlamaRMSNorm.forward")

# Apply F.rms_norm(weight=None) monkey-patch on every LlamaRMSNorm module
print("\napplying F.rms_norm(weight=None) monkey-patch...")
n_patched = 0
for m in model.modules():
    if isinstance(m, LlamaRMSNorm):
        shape = (m.weight.shape[-1],)
        eps = m.variance_epsilon
        def fwd(self, x, _shape=shape, _eps=eps):
            input_dtype = x.dtype
            x = x.to(torch.float32)
            out = F.rms_norm(x, _shape, None, _eps)
            return out.to(input_dtype)
        m.forward = fwd.__get__(m)
        n_patched += 1
print(f"patched {n_patched} LlamaRMSNorm modules")

# Variant B: F.rms_norm(weight=None)
ms_b, tps_b, sample_b = benchmark("B: F.rms_norm(weight=None), fp32 cast preserved")

hf_delta_pct = 100 * (tps_b / tps_a - 1)
print(f"\nrealized speedup B vs A: {hf_delta_pct:+.2f}%")

## Result

In [ ]:
print("=" * 64)
print(f"FlashNorm realized speedup in HuggingFace Transformers")
print(f"GPU: {GPU_NAME}")
print(f"Model: {MODEL_ID}")
print(f"Protocol: bf16, {N_TRIALS} trials, {MAX_TOKENS}-token greedy decode, {N_WARMUP} warmup")
print("=" * 64)
print(f"  A (HF default LlamaRMSNorm.forward):     {tps_a:6.2f} tok/s  ({ms_a:.2f}s)")
print(f"  B (F.rms_norm(weight=None) monkey-patch): {tps_b:6.2f} tok/s  ({ms_b:.2f}s)")
print(f"\n  Realized speedup B vs A: {hf_delta_pct:+.2f}%")
print("=" * 64)

out = {
    "gpu":         GPU_NAME,
    "model":       MODEL_ID,
    "prompt":      PROMPT,
    "dtype":       "bfloat16",
    "max_tokens":  MAX_TOKENS,
    "n_trials":    N_TRIALS,
    "n_warmup":    N_WARMUP,
    "variant_a":   {"label": "HF default LlamaRMSNorm.forward",          "median_s": ms_a, "median_tps": tps_a, "sample": sample_a},
    "variant_b":   {"label": "F.rms_norm(weight=None) monkey-patch",      "median_s": ms_b, "median_tps": tps_b, "sample": sample_b},
    "delta_pct":   hf_delta_pct,
}
with open("flashnorm_hf_results.json", "w") as f:
    json.dump(out, f, indent=2)
print("\nresults saved to flashnorm_hf_results.json")

try:
    from google.colab import files
    files.download("flashnorm_hf_results.json")
except ImportError:
    print("not in Colab; file in working directory")

## Notes

- This measures *kernel-choice* speedup, not folding speedup: the checkpoint is already FlashNorm-folded in both variants. The gap that variant B closes is the per-channel multiply by the all-ones weight tensor that variant A still performs unconditionally.
- The gain composes with the RMSNorm decode-time fraction reported separately (24.6% at bf16). The realized gain (~13%) recovers about half of that ceiling via kernel choice alone, with the rest gated on a fused kernel that performs the normalize and the (skipped) multiply in a single launch.
- For production use, the runtime monkey-patch demonstrated here can be replaced by a flag on `LlamaRMSNorm` upstream in transformers; small follow-up but not a blocker for the measurement.